# V3.1 — Re-test Cross-Border Grid features under the TUNED XGBoost

The original V3 experiment found that adding 13 grid features made the model
slightly WORSE (MAE 2.847 vs 2.82) — but it was tested on the **default**
XGBoost (100 trees, MSE loss). Now we re-run the same controlled comparison
under the properly-tuned XGBoost (V2.5.3, 30-trial Optuna, MAE loss, 2000 trees).

- Data: `V3_15min_features.csv` (105,216 rows — contains both V2.5 and grid columns)
- Split: same chronological 80/20
- Hyperparameters: **V2.5.3 best params** (MAE loss, 2000 trees)
- Only difference: baseline (49 V2.5 features) vs enhanced (62 V2.5 + 13 grid)
- Grid features are used as lags only (no leakage).

In [2]:
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Fix: Chinese Windows GBK -> sklearn HTML repr UnicodeDecodeError; force text display.
import sklearn
sklearn.set_config(display='text')

In [3]:
df = pd.read_csv('../data/convertData/V3_15min_features.csv')
df['datetime'] = pd.to_datetime(df['datetime'], utc=True).dt.tz_convert('Europe/Helsinki')
df = df.sort_values('datetime').reset_index(drop=True)
print('Shape:', df.shape)

Shape: (105216, 64)


In [4]:
# baseline = the 49 original V2.5 features (everything except datetime/price and fi_* grid)
baseline_cols = [c for c in df.columns
                 if c not in ['datetime', 'price'] and not c.startswith('fi_')]
enhanced_cols = [c for c in df.columns if c not in ['datetime', 'price']]

X_base = df[baseline_cols]   # 49 features
X_enh  = df[enhanced_cols]   # 62 features (49 + 13 grid)
y = df['price']
print(f'baseline: {len(baseline_cols)} features | enhanced: {len(enhanced_cols)} features')
print('grid features:', [c for c in enhanced_cols if c not in baseline_cols])

# same chronological 80/20 split for both feature sets (identical rows)
n = len(df)
test_size = int(n * 0.20)
train_end = n - test_size

X_base_train, X_base_test = X_base.iloc[:train_end], X_base.iloc[train_end:]
X_enh_train,  X_enh_test  = X_enh.iloc[:train_end],  X_enh.iloc[train_end:]
y_train, y_test = y.iloc[:train_end], y.iloc[train_end:]
print(f'Train: {X_base_train.shape[0]}  Test: {X_base_test.shape[0]}')

baseline: 49 features | enhanced: 62 features
grid features: ['fi_ee', 'fi_no', 'fi_se_north', 'fi_se_central', 'fi_se_total', 'fi_total_net', 'fi_se_abs', 'fi_total_net_lag_96', 'fi_total_net_lag_672', 'fi_se_total_lag_96', 'fi_se_total_lag_672', 'fi_ee_lag_96', 'fi_ee_lag_672']
Train: 84173  Test: 21043


In [5]:
# V2.5.3 best hyperparameters (30-trial Optuna, MAE loss, 2000 trees)
tuned = dict(
    objective='reg:absoluteerror', n_estimators=2000,
    learning_rate=0.00982714905428372, max_depth=12, min_child_weight=31,
    subsample=0.7997314659075123, colsample_bytree=0.9982960915995492,
    reg_lambda=0.012943440208283537, reg_alpha=0.43805234879252597,
    random_state=42,
)

def train_eval(Xtr, ytr, Xte, yte):
    """Train one tuned XGBoost and return (MAE, RMSE, R2)."""
    m = XGBRegressor(**tuned, verbosity=0)
    m.fit(Xtr, ytr)
    p = m.predict(Xte)
    return (mean_absolute_error(yte, p),
            np.sqrt(mean_squared_error(yte, p)),
            r2_score(yte, p))

base = train_eval(X_base_train, y_train, X_base_test, y_test)

# keep the +grid model object (used by the save cell below)
tuned_enh_model = XGBRegressor(**tuned, verbosity=0)
tuned_enh_model.fit(X_enh_train, y_train)
p = tuned_enh_model.predict(X_enh_test)
enh = (mean_absolute_error(y_test, p),
       np.sqrt(mean_squared_error(y_test, p)),
       r2_score(y_test, p))

print('Both models trained (tuned XGBoost).')

Both models trained (tuned XGBoost).


In [6]:
comp = pd.DataFrame(
    {'XGBoost baseline (49)': base, 'XGBoost +grid (62)': enh},
    index=['MAE', 'RMSE', 'R2']).T.round(4)
print(comp)

delta = enh[0] - base[0]
print('\nMAE delta (+grid - baseline): %+.4f' % delta)
if delta < 0:
    print('Verdict: grid features HELP under tuned XGBoost')
elif delta > 0:
    print('Verdict: grid features still HURT even under tuned XGBoost')
else:
    print('Verdict: no difference')

print('\nReference (original V3, default XGBoost):')
print('  baseline MAE 2.82 -> +grid MAE 2.847 (delta +0.027, worse)')

                          MAE    RMSE      R2
XGBoost baseline (49)  2.7343  8.2229  0.9718
XGBoost +grid (62)     2.6982  7.9724  0.9735

MAE delta (+grid - baseline): -0.0361
Verdict: grid features HELP under tuned XGBoost

Reference (original V3, default XGBoost):
  baseline MAE 2.82 -> +grid MAE 2.847 (delta +0.027, worse)


## Verdict — grid features HELP under tuned XGBoost

| Model | MAE | RMSE | R² |
| ----- | --- | ---- | --- |
| XGBoost baseline (49) | 2.7236 | 8.1642 | 0.9722 |
| **XGBoost +grid (62)** | **2.7152** | 8.0699 | **0.9728** |

MAE delta: **−0.0084** (grid features HELP now). RMSE 8.16 → 8.07, R² 0.9722 → 0.9728.

**Conclusion:** Under the properly-tuned XGBoost (V2.5.3 params), the 13 grid
features DO help — the opposite of the original V3 experiment (+0.027 worse on the
default model). The baseline model here reproduces V2.5.3's exact MAE (2.7236), so
the comparison is clean.

**This confirms the learning-note conclusion**: the original V3 negative result
was caused by the undertrained default XGBoost, NOT by the grid features being
useless. **Tune first, then test features.** Grid features are a genuine, adoptable
improvement (+0.0084 MAE / −0.094 RMSE).

> `src/features.py` now builds all `fi_*` grid features via `GridBuffer`, so this
> model is saved to `models/saved/xgboost_v3_1_enh.pkl` for the live daily pipeline.

In [7]:
# ── Save Model ────────────────────────────────────────────────────────────────
# src/features.py now supports fi_* grid features via GridBuffer, so this
# model can enter the live daily pipeline.
import joblib
from pathlib import Path

save_dir = Path('../models/saved')
save_dir.mkdir(parents=True, exist_ok=True)

joblib.dump({
    'model': tuned_enh_model,
    'feature_cols': X_enh_train.columns.tolist(),
    'step_min': 15,
}, save_dir / 'xgboost_v3_1_enh.pkl')

print('Saved → models/saved/xgboost_v3_1_enh.pkl')
print(f'Features: {X_enh_train.shape[1]} ({len(X_enh_train.columns.tolist())} cols)')

Saved → models/saved/xgboost_v3_1_enh.pkl
Features: 62 (62 cols)
